# ECCOFS `avg`: vertical temperature section off New Jersey with xroms

Opens the whole-column best-estimate virtual Icechunk store (`eccofs-avg-best-nosub`) on [Source Cooperative](https://source.coop), cuts out the mid-Atlantic Bight, lets [xroms](https://xroms.readthedocs.io) compute the terrain-following depths (`z_rho`), and draws a vertical section of temperature along an offshore transect from the New Jersey coast across the shelf break.

A section reads every vertical level in the box, so this notebook uses the `-nosub` store, where all 50 levels of a tile are one chunk. It reads the section data about 2x faster than the default `eccofs-avg-best` store, whose chunks are single levels and are better for maps and point time series (see the README, "Chunking").

No credentials are needed. It needs icechunk 2.x (Python 3.12+) and `xroms` 0.6.2 with `xgcm` 0.8.1 (`conda install -c conda-forge xroms=0.6.2 xgcm=0.8.1`); newer xgcm releases break xroms' grid parsing.

In [ ]:
# %pip install -q "xroms==0.6.2" "xgcm==0.8.1" hvplot datashader geoviews cartopy icechunk zarr scipy

In [ ]:
import hvplot.xarray  # noqa: F401
import cartopy.crs as ccrs
import geoviews as gv
import holoviews as hv
import icechunk
import numpy as np
import xarray as xr
import xroms
import zarr
from scipy.interpolate import LinearNDInterpolator

zarr.config.set({"async.concurrency": 32})

# whole-column chunks (all 50 levels of a tile): the better layout for sections. For maps and time series use eccofs-avg-best
URL = "https://data.source.coop/ocean-icechunks/noaa-eccofs/eccofs-avg-best-nosub"
SOURCE = "s3://noaa-nos-eccofs-pds/"   # public bucket holding the NetCDF bytes

# transect: off Atlantic City, NJ, running south-east across the shelf and slope (edit me)
START = (-74.30, 39.30)   # lon, lat
END = (-72.00, 37.20)
WHEN = "2026-11-15 12:00"

In [ ]:
# the NetCDF bytes live in NOAA's bucket, outside the store: authorize that virtual chunk container (anonymous)
repo = icechunk.Repository.open(
    icechunk.http_storage(URL),
    authorize_virtual_chunk_access=icechunk.containers_credentials({SOURCE: icechunk.s3_anonymous_credentials()}),
)
ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks={})
print(f"{ds.sizes['time']} times: {ds.time.values[0]} to {ds.time.values[-1]}")

## Cut out the region and one time

The full grid is 1443 x 1667 x 50 levels, so subset first. Boolean indexers on the `rho` dimensions select the box. The staggered `u`/`v`/`psi` dimensions get the matching indexers (one fewer point than `rho` across the staggering), which xgcm requires.

In [ ]:
box = ((ds.lon_rho > -76) & (ds.lon_rho < -70) & (ds.lat_rho > 36) & (ds.lat_rho < 41)).load()
jm, im = box.any("xi_rho").values, box.any("eta_rho").values   # rows / columns that touch the box
# u, v and psi points sit between rho points, so keep those whose two rho neighbours are both kept
jm_in, im_in = jm[:-1] & jm[1:], im[:-1] & im[1:]

region = ds.isel(
    eta_rho=jm, xi_rho=im,
    eta_u=jm, xi_u=im_in,
    eta_v=jm_in, xi_v=im,
    eta_psi=jm_in, xi_psi=im_in,
)
keep = ["temp", "u", "v", "zeta", "h", "Cs_r", "Cs_w", "pm", "pn", "angle", "f", "mask_rho", "mask_u", "mask_v", "mask_psi"]
sub = region[keep].sel(time=WHEN, method="nearest")
print("time:", sub.time.values, "| subset:", dict(sub.sizes))

## Scalar parameters

xroms builds `z_rho` from `Vtransform`, `Vstretching`, `theta_s`, `theta_b`, `hc` (and needs `spherical`). The stores first dropped these model scalars; they are now written back by `scripts/add_scalars.py`. Until a store has them, the notebook fills them in with the values from the ECCOFS NetCDF files (the same ones `add_scalars.py` copies).

In [ ]:
ECCOFS_SCALARS = dict(Vtransform=2, Vstretching=4, theta_s=7.0, theta_b=2.0, Tcline=250.0, hc=250.0, rho0=1025.0, spherical=1)
missing = {k: v for k, v in ECCOFS_SCALARS.items() if k not in ds}
print("in store:", [k for k in ECCOFS_SCALARS if k in ds], "| filled in here:", list(missing))
for k in ECCOFS_SCALARS:
    sub = sub.assign({k: ds[k].load()}) if k in ds else sub.assign({k: missing[k]})

## xroms dataset and depths

`roms_dataset` builds the xgcm grid and the depth fields. Two things the ECCOFS store needs: the SGRID `Conventions` attribute is dropped (xroms builds its own grid from the dimension names) and the arrays are rechunked to single chunks along the horizontal dimensions, which xgcm requires for staggered interpolation.

In [ ]:
sub.attrs["Conventions"] = "CF-1.4"
sub = sub.chunk({d: -1 for d in sub.dims})
rds, xgrid = xroms.roms_dataset(sub)
z_rho = rds.z_rho
assert float(z_rho.isel(s_rho=-1).max()) < 5 and float(z_rho.isel(s_rho=0).min()) < -3000, "unexpected depths"
print(f"z_rho: surface {float(z_rho.isel(s_rho=-1).mean()):.2f} m mean, deepest {float(z_rho.isel(s_rho=0).min()):.0f} m")

## Transect

Points are spaced about 3 km (the grid size) along the great circle between `START` and `END`. Each point is located in the grid's (eta, xi) index space with a Delaunay interpolation of longitude/latitude, and the fields are bilinearly interpolated there, so the curvilinear grid needs no projection. Land is masked before interpolating.

In [ ]:
def haversine_km(lon1, lat1, lon2, lat2):
    p = np.pi / 180
    a = np.sin((lat2 - lat1) * p / 2) ** 2 + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2
    return 12742 * np.arcsin(np.sqrt(a))

total_km = haversine_km(*START, *END)
n = int(total_km / 3) + 1
lons = np.linspace(START[0], END[0], n)
lats = np.linspace(START[1], END[1], n)
dist = xr.DataArray(np.linspace(0, total_km, n), dims="dist", attrs={"long_name": "distance along section", "units": "km"})

lon2d, lat2d = rds.lon_rho.load().values, rds.lat_rho.load().values
jj, ii = np.meshgrid(rds.eta_rho.values, rds.xi_rho.values, indexing="ij")
pts = np.c_[lon2d.ravel(), lat2d.ravel()]
fj = LinearNDInterpolator(pts, jj.ravel())(lons, lats)
fi = LinearNDInterpolator(pts, ii.ravel())(lons, lats)
assert not (np.isnan(fj).any() or np.isnan(fi).any()), "transect leaves the subset"
fj, fi = xr.DataArray(fj, dims="dist"), xr.DataArray(fi, dims="dist")
print(f"{n} points over {total_km:.0f} km")

In [ ]:
%%time
mask = rds.mask_rho
temp = rds.temp.where(mask == 1).load()
z = z_rho.load()
h = rds.h.where(mask == 1).load()

def along(da):
    return da.interp(eta_rho=fj, xi_rho=fi).assign_coords(dist=dist)

sec_t, sec_z, sec_h = along(temp), along(z), along(h)

## Section

Temperature is drawn on the true (moving) depth of each level, `z_rho`, with the sea floor from `h`. Look for the shelf water inshore of the shelf break, the thermocline sloping down across the slope, and the warm Gulf Stream-influenced water offshore.

In [ ]:
sec = xr.Dataset({"temp": sec_t, "z_rho": sec_z}).rename({"s_rho": "level"}).drop_vars(["s_rho"], errors="ignore")
sec = sec.assign_coords(dist=dist, km=(("level", "dist"), np.broadcast_to(dist.values, sec.z_rho.shape).copy()))   # quadmesh wants x and y on the same dims
sec.temp.attrs.update(long_name="potential temperature", units="degC")

section = sec.temp.hvplot.quadmesh(
    x="km", y="z_rho", rasterize=True, cmap="turbo", colorbar=True,
    xlabel="distance from the coast (km)", ylabel="depth (m)",
    title=f"ECCOFS temperature, {np.datetime_as_string(sub.time.values, unit='h')} UTC",
    frame_width=700, frame_height=350, ylim=(-1000, 5), xlim=(0, float(dist[-1])),
)
floor = hv.Area((dist.values, -sec_h.values, np.full(n, -4000.0)), kdims="dist", vdims=["bottom", "floor"]).opts(color="#4d4d4d", line_width=0)
section * floor

In [ ]:
sst = temp.isel(s_rho=-1).rename("sst").assign_coords(lon_rho=(("eta_rho", "xi_rho"), lon2d), lat_rho=(("eta_rho", "xi_rho"), lat2d))
surface = sst.hvplot.quadmesh(
    x="lon_rho", y="lat_rho", rasterize=True, geo=True, tiles="OSM", cmap="turbo",
    frame_width=450, title="Surface temperature (degC) and transect",
)
# geoviews path with a CRS, so the degrees are projected like the map (a plain hv.Path would be read as metres, near 0,0)
track = np.c_[lons, lats]
line = gv.Path([track], crs=ccrs.PlateCarree()).opts(color="white", line_width=3) * gv.Path([track], crs=ccrs.PlateCarree()).opts(color="black", line_width=1)
surface * line

Depths down to 1000 m are shown so the shelf is readable; use `ylim=(-3500, 5)` to see the full slope.